# Compare heat-wave trends across Western cities

**Terrain advanced · Python project · NOAA Climate Data**

Pull daily maximum temperature for five Western U.S. cities, define heat waves as **3+ consecutive days above each city's 1980s extreme**, and compare the **1980s** vs **2010s**.

---

## Learning objectives

By the end, you should be able to:

- Download daily climate data for multiple stations
- Define an **event-based** metric (not just a mean trend)
- Use a **city-specific baseline threshold** for fair comparison
- Compare decades and explain why extremes matter for health

**Time:** ~2.5 hr · **Dataset:** [NOAA Climate Data explainer](../explainer.html?id=noaa-climate)

---

## For mentors

- **Prerequisites:** [Hometown temperature trend](hometown_temperature_trend.ipynb) helps.
- **Live data:** Default pulls daily TMAX from **NOAA NCEI** (10 decade-sized requests, ~2–5 min).
- **Watch for:** Students using one fixed °F cutoff for Phoenix and Portland — we use each city's **1980s 95th percentile** instead.
- **Denver station note:** `USW00023062` has full 1980s coverage; `USW00003017` does not.

**Suggested flow:** Read each section → student runs the cell → use *Mentor check-in* before continuing.

---

**Goal:** Set cities, decades, and heat-wave rules.

**Concept:** A **heat wave** here = at least `MIN_CONSECUTIVE_DAYS` days in a row above a threshold. Threshold = each city's **95th percentile daily TMAX in the 1980s** (a baseline "extreme" for that place).

**Mentor check-in:** *Why not use 95°F for every city?*

In [ ]:
from io import StringIO
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import requests

CITIES = {
    "USW00023174": "Los Angeles (LAX)",
    "USW00023183": "Phoenix",
    "USW00023188": "Las Vegas",
    "USW00024229": "Portland",
    "USW00023062": "Denver",
}
EARLY_DECADE = (1980, 1989)
LATE_DECADE = (2010, 2019)
MIN_CONSECUTIVE_DAYS = 3
NOAA_URL = "https://www.ncei.noaa.gov/access/services/data/v1"

**Goal:** Load daily TMAX for all cities.

**Concept:** NOAA stores TMAX in **tenths of °C**. We fetch **one city + one decade at a time** (more reliable than one giant request).

**You should see:** ~36,000 rows from live NOAA and five cities listed.

**If something breaks:** NOAA can be slow — wait for all 10 fetch messages. Check internet access in Colab.

In [ ]:
def fetch_decade(station_id, start, end):
    params = {
        "dataset": "daily-summaries",
        "stations": station_id,
        "startDate": start,
        "endDate": end,
        "dataTypes": "TMAX",
        "format": "csv",
    }
    resp = requests.get(NOAA_URL, params=params, timeout=180)
    resp.raise_for_status()
    return pd.read_csv(StringIO(resp.text))

frames = []
for sid, label in CITIES.items():
    for decade in (EARLY_DECADE, LATE_DECADE):
        start = f"{decade[0]}-01-01"
        end = f"{decade[1]}-12-31"
        chunk = fetch_decade(sid, start, end)
        chunk["city"] = label
        frames.append(chunk)
        print(f"Fetched {label} {decade[0]}s: {len(chunk):,} rows from NOAA")

daily = pd.concat(frames, ignore_index=True)
daily["DATE"] = pd.to_datetime(daily["DATE"])
daily["TMAX_F"] = pd.to_numeric(daily["TMAX"], errors="coerce") / 10 * 9 / 5 + 32
daily = daily.dropna(subset=["TMAX_F"])
print(f"\nTotal rows: {len(daily):,} · Cities: {sorted(daily['city'].unique())}")
daily.head()

**Goal:** Count heat-wave **days** in each decade.

**Concept:** First lock in each city's **1980s 95th percentile** threshold. Then count days that fall inside runs of `MIN_CONSECUTIVE_DAYS` or more above that threshold.

**You should see:** Every city has higher (or equal) heat-wave days in the 2010s vs 1980s.

**Mentor check-in:** *Is "more hot days" the same as "higher average temperature"?*

In [ ]:
def heat_wave_days(series_f, threshold, min_days=MIN_CONSECUTIVE_DAYS):
    hot = series_f >= threshold
    run_id = (hot != hot.shift()).cumsum()
    total = 0
    for _, run in pd.DataFrame({"hot": hot, "run_id": run_id}).groupby("run_id"):
        if run["hot"].iloc[0] and len(run) >= min_days:
            total += len(run)
    return total

rows = []
for city, g in daily.groupby("city"):
    base = g[(g["DATE"].dt.year >= EARLY_DECADE[0]) & (g["DATE"].dt.year <= EARLY_DECADE[1])]
    threshold = base["TMAX_F"].quantile(0.95)
    for label, decade in [("1980s", EARLY_DECADE), ("2010s", LATE_DECADE)]:
        sub = g[(g["DATE"].dt.year >= decade[0]) & (g["DATE"].dt.year <= decade[1])].sort_values("DATE")
        days = heat_wave_days(sub["TMAX_F"], threshold)
        rows.append({
            "city": city,
            "decade": label,
            "threshold_f": round(threshold, 1),
            "heat_wave_days": days,
        })

summary = pd.DataFrame(rows)
summary

**Goal:** Plot decade comparison.

**You should see:** Grouped bars — 1980s vs 2010s for each city.

**Mentor check-in:** *Which city saw the largest jump? Does that match what you know about drought and urban heat?*

In [ ]:
pivot = summary.pivot(index="city", columns="decade", values="heat_wave_days")
pivot = pivot.loc[sorted(pivot.index)]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(pivot))
width = 0.35
ax.bar(x - width/2, pivot["1980s"], width, label="1980s", color="#5B8BB0")
ax.bar(x + width/2, pivot["2010s"], width, label="2010s", color="#C45C5C")
ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=15, ha="right")
ax.set_ylabel("Heat-wave days per decade")
ax.set_title("Heat-wave days above each city's 1980s extreme (95th pct TMAX)")
ax.legend()
plt.tight_layout()
plt.show()

pivot.assign(change=pivot["2010s"] - pivot["1980s"])

---

## Final takeaway

Student writes 3–4 sentences. **Mentor prompts:**

1. Did every city show more heat-wave days in the 2010s?
2. Why use a **city-specific** threshold instead of one national cutoff?
3. What health or infrastructure risks do longer heat waves create?

**Mentor rubric:** Names the metric · Cites decades · Mentions at least one limitation (one station ≠ whole metro; definition choices matter)

**Extension:** Change `MIN_CONSECUTIVE_DAYS` to 5 or use the 99th percentile threshold — how do results shift?

**Next:** [NOAA explainer](../explainer.html?id=noaa-climate) · [Hometown temperature](hometown_temperature_trend.ipynb)